# Preparación, limpieza y estructuración de los datos del proyecto

**Actividad:** GA1-220501094-01-AA1-EV07 – Preparación, limpieza y estructuración de datos del proyecto individual con Pandas
**Aprendiz:** Pablo Enrique Fuenmayor Osuna · Servicio Nacional de Aprendizaje (SENA)
**Proyecto:** Tecnologías y salarios de desarrolladores de software

**Pregunta analítica:**

> ¿Qué lenguajes de programación y frameworks se asocian con un salario anual mayor entre desarrolladores profesionales de tiempo completo, y cuánto cambia esa diferencia según los años de experiencia y la región, entre 2022 y 2025?

**Pregunta orientadora de la actividad:** ¿cómo preparar y estructurar los datos del proyecto para obtener información confiable que permita responder la pregunta analítica?

Este cuaderno aplica las acciones propuestas en el diagnóstico de la actividad anterior ([`reports/data_quality.md`](../reports/data_quality.md) y [`data_quality.ipynb`](data_quality.ipynb)). Sigue el orden de las lecciones 2, 3 y 4 del material *Preparar y estructurar datos con Python*:

| Sección | Contenido |
|---|---|
| 0 | Configuración, carga de los datos originales y estado inicial |
| Lección 2 | Limpieza y estandarización: duplicados, tipos, texto y categorías, nulos, atípicos, pipeline con `pipe()` y validación |
| Lección 3 | Filtrado, ordenamiento y agrupación: población de interés e indicadores |
| Lección 4 | Combinación y estructuración: `concat`, `melt`, `merge` validados y exportación a `data/processed/` |
| 5 | Comparación de la calidad antes y después |
| 6 | Conclusiones |

Los archivos de `data/raw/` se leen pero **no se modifican**. Todo lo que produce el cuaderno se guarda en `data/processed/`.

## 0. Configuración y carga de los datos originales

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', '{:,.2f}'.format)

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW = RAIZ / 'data' / 'raw'
PROCESSED = RAIZ / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)

ANIOS = [2022, 2023, 2024, 2025]
ARCHIVOS = [f'so_survey_{a}.csv' for a in ANIOS] + ['gdp_per_capita_owid.csv', 'gdp_per_capita_ppp_wb.csv']
faltantes = [f for f in ARCHIVOS if not (RAW / f).exists()]
assert not faltantes, f'Faltan archivos en data/raw: {faltantes}. Ejecuta: python scripts/download_data.py'

print('Python', sys.version.split()[0], '| pandas', pd.__version__, '| numpy', np.__version__)

Python 3.14.7 | pandas 3.1.0 | numpy 2.5.4


De cada encuesta se cargan solo las **11 columnas** que necesita la pregunta analítica (`usecols`). Las demás, como las preguntas sobre IA que hacen crecer la encuesta de 79 a 172 columnas, no se usan. Así también quedan fuera las dos columnas vacías de 2022 (`VCHosting*`) que señaló el diagnóstico. Se usa una función en `usecols` porque 2025 ya no tiene `YearsCodePro`.

In [2]:
COLUMNAS_SO = ['ResponseId', 'MainBranch', 'Employment', 'Country', 'Age',
               'YearsCode', 'YearsCodePro', 'WorkExp',
               'LanguageHaveWorkedWith', 'WebframeHaveWorkedWith', 'ConvertedCompYearly']

crudos = {anio: pd.read_csv(RAW / f'so_survey_{anio}.csv', usecols=lambda c: c in COLUMNAS_SO)
          for anio in ANIOS}
owid_crudo = pd.read_csv(RAW / 'gdp_per_capita_owid.csv')
bm_crudo = pd.read_csv(RAW / 'gdp_per_capita_ppp_wb.csv', skiprows=4)   # 4 líneas de metadatos

tamanos = {f'so_{a}': crudos[a].shape for a in ANIOS} | {'gdp_owid': owid_crudo.shape, 'gdp_wb': bm_crudo.shape}
pd.DataFrame(tamanos, index=['filas', 'columnas']).T

,filas,columnas
so_2022,73268,11
so_2023,89184,11
so_2024,65437,11
so_2025,49191,10
gdp_owid,7445,5
gdp_wb,266,71


### 0.1 Estado inicial de la calidad

Para comparar al final, se resume la calidad de las encuestas **antes** de procesarlas con los mismos indicadores que se medirán después. La función `perfil_calidad` recibe los nombres de las columnas porque cambian después de la limpieza.

In [3]:
def perfil_calidad(df, *, salario, lenguajes, frameworks, experiencia, con_pib):
    """Resume la calidad de una encuesta con indicadores comparables antes y después."""
    def etiquetas(col):
        return df[col].dropna().str.split(';').explode().str.strip().nunique()
    return {
        'filas': len(df),
        'columnas': df.shape[1],
        'filas duplicadas': int(df.duplicated().sum()),
        'salario nulo (%)': round(df[salario].isna().mean() * 100, 1),
        'salario mínimo (USD)': df[salario].min(),
        'salario máximo (USD)': df[salario].max(),
        'salario mediano (USD)': df[salario].median(),
        'salario medio (USD)': round(df[salario].mean(), 0),
        'etiquetas de lenguajes': etiquetas(lenguajes),
        'etiquetas de frameworks': etiquetas(frameworks),
        'experiencia: tipo de dato': str(df[experiencia].dtype),
        'experiencia nula (%)': round(df[experiencia].isna().mean() * 100, 1),
        'respuestas sin PIB del país (%)': round((~con_pib).mean() * 100, 1),
    }

paises_owid = set(owid_crudo['entity'])
crudo_total = pd.concat(crudos.values(), ignore_index=True)

def perfil_crudo(df):
    experiencia = 'YearsCodePro' if df['YearsCodePro'].notna().any() else 'WorkExp'
    return perfil_calidad(df, salario='ConvertedCompYearly', lenguajes='LanguageHaveWorkedWith',
                          frameworks='WebframeHaveWorkedWith', experiencia=experiencia,
                          con_pib=df['Country'].isin(paises_owid))

calidad_antes = pd.DataFrame({**{a: perfil_crudo(crudos[a].reindex(columns=COLUMNAS_SO)) for a in ANIOS},
                              'total': perfil_crudo(crudo_total)})
calidad_antes

,2022,2023,2024,2025,total
filas,73268,89184,65437,49191,277080
columnas,11,11,11,11,11
filas duplicadas,0,0,0,0,0
salario nulo (%),48.00,46.20,64.20,51.30,51.80
salario mínimo (USD),1.00,1.00,1.00,1.00,1.00
salario máximo (USD),"50,000,000.00","74,351,432.00","16,256,603.00","50,000,000.00","74,351,432.00"
salario mediano (USD),"67,845.00","74,963.00","65,000.00","75,320.00","71,451.00"
salario medio (USD),"170,761.00","103,110.00","86,155.00","101,762.00","119,188.00"
etiquetas de lenguajes,42,51,49,42,58
etiquetas de frameworks,25,34,36,28,42


En la columna `total` se ven mejor dos problemas que no aparecen dentro de un solo año: al juntar los cuatro años hay **58 etiquetas de lenguajes y 42 de frameworks** (contando ya sin espacios sobrantes), y algunas son la misma tecnología escrita de otra forma. Además, el **36,5 % de las respuestas** no encuentra su país en la fuente de PIB, ya sea porque el nombre no coincide o porque el país está vacío.

## Lección 2: Limpieza y estandarización de datos

Se sigue el orden del material: **duplicados, selección y nombres, tipos, texto y categorías, nulos, atípicos e incoherencias**, y al final un pipeline con `pipe()` que registra cada paso y una validación con `assert`. Para cada problema hay tres opciones (**eliminar, corregir o conservar**) y la decisión depende de qué significa el dato.

En esta lección **no se eliminan filas**: solo se corrigen valores. La selección de la población de interés es un filtro y se hace en la Lección 3.

### 2.1 Duplicados

In [4]:
for anio, df in crudos.items():
    print(anio, '| filas duplicadas:', df.duplicated().sum(), '| ResponseId único:', df['ResponseId'].is_unique)

print('OWID: pares país-año repetidos:', owid_crudo.duplicated(['entity', 'year']).sum())
print('Banco Mundial: códigos de país repetidos:', bm_crudo['Country Code'].duplicated().sum())

2022 | filas duplicadas: 0 | ResponseId único: True
2023 | filas duplicadas: 0 | ResponseId único: True
2024 | filas duplicadas: 0 | ResponseId único: True
2025 | filas duplicadas: 0 | ResponseId único: True
OWID: pares país-año repetidos: 0
Banco Mundial: códigos de país repetidos: 0


No hay duplicados exactos. Pero el identificador hace que dos respuestas idénticas parezcan distintas, así que se buscan **duplicados implícitos**: personas con salario que respondieron exactamente lo mismo en las 10 columnas sin contar `ResponseId`. Si aparecen, se comprueba si también coinciden en **todas** las columnas de la encuesta original (hasta 172), cargando el archivo completo.

In [5]:
sin_id = [c for c in COLUMNAS_SO if c != 'ResponseId']
filas = []
for anio, df in crudos.items():
    columnas = [c for c in sin_id if c in df.columns]
    con_salario = df[df['ConvertedCompYearly'].notna()]
    completo = pd.read_csv(RAW / f'so_survey_{anio}.csv', low_memory=False)
    completo = completo[completo['ConvertedCompYearly'].notna()].drop(columns='ResponseId')
    filas.append({'año': anio,
                  'con salario': len(con_salario),
                  'iguales en las 10 columnas': int(con_salario.duplicated(subset=columnas).sum()),
                  'iguales en todas las columnas': int(completo.duplicated().sum())})
    del completo
duplicados_implicitos = pd.DataFrame(filas).set_index('año')
duplicados_implicitos

,con salario,iguales en las 10 columnas,iguales en todas las columnas
año,,,
2022,38071,1,0
2023,48019,3,0
2024,23435,2,0
2025,23947,9,0


Cada año hay entre 1 y 9 respuestas con salario iguales a otra en las 10 columnas, pero **ninguna es igual en todas las columnas** de la encuesta. Son personas distintas con un perfil corto parecido (mismo país, edad, experiencia y un salario redondo), no envíos repetidos.

**Decisión: conservar.** Borrarlas eliminaría respuestas reales. En las fuentes de PIB tampoco hay duplicados.

### 2.2 Selección de columnas y nombres estandarizados

Las columnas se renombran en inglés y en `snake_case`, igual que la estructura del repositorio, y se agrega el año. `seleccionar_columnas` deja las cuatro encuestas con **las mismas columnas en el mismo orden**, condición necesaria para unirlas después con `concat`. En 2025 `years_code_pro` queda vacía porque la pregunta no existe.

In [6]:
NOMBRES = {'ResponseId': 'response_id', 'MainBranch': 'main_branch', 'Employment': 'employment',
           'Country': 'country', 'Age': 'age', 'YearsCode': 'years_code',
           'YearsCodePro': 'years_code_pro', 'WorkExp': 'work_exp',
           'LanguageHaveWorkedWith': 'languages', 'WebframeHaveWorkedWith': 'frameworks',
           'ConvertedCompYearly': 'salary_usd'}
COLUMNAS_BASE = ['year', *NOMBRES.values()]


def seleccionar_columnas(df, anio):
    """Renombra, agrega el año y deja las columnas en un orden común."""
    df = df.rename(columns=NOMBRES).assign(year=anio)
    if 'years_code_pro' not in df.columns:      # 2025 ya no pregunta YearsCodePro
        df['years_code_pro'] = np.nan
    return df[COLUMNAS_BASE]


seleccionar_columnas(crudos[2025], 2025).head(3)

,year,response_id,main_branch,employment,country,age,years_code,years_code_pro,work_exp,languages,frameworks,salary_usd
0,2025,1,I am a developer by profession,Employed,Ukraine,25-34 years old,14.00,NaN,8.00,Bash/Shell (all shells);Dart;SQL,NaN,"61,256.00"
1,2025,2,I am a developer by profession,Employed,Netherlands,25-34 years old,10.00,NaN,2.00,Java,Spring Boot,"104,413.00"
2,2025,3,I am a developer by profession,"Independent contractor, freelancer, or self-em...",Ukraine,35-44 years old,12.00,NaN,10.00,Dart;HTML/CSS;JavaScript;TypeScript,Next.js;Node.js;React,"53,061.00"


### 2.3 Tipos de datos

El diagnóstico encontró que la experiencia se lee como texto en 2022–2024 y como número en 2025:

In [7]:
pd.DataFrame({a: crudos[a].dtypes.reindex(['YearsCode', 'YearsCodePro', 'WorkExp', 'ConvertedCompYearly']).astype(str)
              for a in ANIOS})

,2022,2023,2024,2025
YearsCode,str,str,str,float64
YearsCodePro,str,str,str,NaN
WorkExp,float64,float64,float64,float64
ConvertedCompYearly,float64,float64,float64,float64


In [8]:
texto = pd.concat([crudos[a][['YearsCode', 'YearsCodePro']] for a in ANIOS if a < 2025]).stack()
no_numericos = texto[pd.to_numeric(texto, errors='coerce').isna()]
no_numericos.value_counts()

Less than 1 year      9614
More than 50 years     953
Name: count, dtype: int64

Solo hay dos etiquetas de texto. Se reemplazan por un número antes de convertir: `Less than 1 year` → 0,5 y `More than 50 years` → 51. Se usa el tipo **`Float64` con mayúscula**, que admite nulos. `errors='coerce'` convertiría en nulo cualquier otra etiqueta inesperada, así que después se comprueba con `assert` que no se perdió ningún valor.

In [9]:
ETIQUETAS_ANIOS = {'Less than 1 year': '0.5', 'More than 50 years': '51'}


def a_numero(serie):
    """Convierte años de experiencia escritos como texto a número decimal nullable."""
    return pd.to_numeric(serie.replace(ETIQUETAS_ANIOS), errors='coerce').astype('Float64')


def corregir_tipos(df):
    df = df.copy()
    for col in ['years_code', 'years_code_pro', 'work_exp']:
        df[col] = a_numero(df[col])
    return df.astype({'year': 'int16', 'response_id': 'int32', 'salary_usd': 'Float64'})


prueba = crudos[2022].pipe(seleccionar_columnas, 2022).pipe(corregir_tipos)
print(prueba[['years_code', 'years_code_pro', 'work_exp', 'salary_usd']].dtypes, '\n')
assert prueba['years_code_pro'].notna().sum() == crudos[2022]['YearsCodePro'].notna().sum(), 'se perdieron valores'
print('Valores no nulos conservados:', prueba['years_code_pro'].notna().sum())

years_code        Float64
years_code_pro    Float64
work_exp          Float64
salary_usd        Float64
dtype: object 

Valores no nulos conservados: 51833


### 2.4 Texto y categorías

**Empleo y rol.** `MainBranch` y `Employment` se convierten en dos columnas booleanas que responden directamente si la persona pertenece a la población de interés. En 2022–2024 `Employment` es de selección múltiple, así que se separa por `;` y se busca la opción exacta. En 2025 la encuesta solo ofrece `Employed`, que se usa como aproximación de tiempo completo (limitación ya declarada en el diagnóstico).

In [10]:
for anio in ANIOS:
    opciones = crudos[anio]['Employment'].str.split(';').explode().value_counts()
    print(anio, opciones.head(3).to_dict())

2022 {'Employed, full-time': 49199, 'Student, full-time': 10932, 'Independent contractor, freelancer, or self-employed': 10721}
2023 {'Employed, full-time': 60899, 'Independent contractor, freelancer, or self-employed': 13988, 'Student, full-time': 11768}
2024 {'Employed, full-time': 45162, 'Independent contractor, freelancer, or self-employed': 10726, 'Student, full-time': 8626}
2025 {'Employed': 33750, 'Independent contractor, freelancer, or self-employed': 6708, 'Student': 4428}


In [11]:
def estandarizar_empleo(df):
    """Crea is_professional y full_time y elimina las columnas de texto originales."""
    df = df.copy()
    objetivo = 'Employed, full-time' if df['year'].iat[0] < 2025 else 'Employed'
    df['is_professional'] = df['main_branch'].eq('I am a developer by profession')
    df['full_time'] = df['employment'].fillna('').str.split(';').map(lambda opciones: objetivo in opciones)
    return df.drop(columns=['main_branch', 'employment'])

**Edad.** Las etiquetas se acortan y se guardan como **categoría ordenada**, para que los grupos salgan en orden de edad. `Prefer not to say` no es una edad: se convierte en nulo.

In [12]:
EDADES = {'Under 18 years old': 'Under 18', '18-24 years old': '18-24', '25-34 years old': '25-34',
          '35-44 years old': '35-44', '45-54 years old': '45-54', '55-64 years old': '55-64',
          '65 years or older': '65+', 'Prefer not to say': None}
ORDEN_EDADES = [e for e in EDADES.values() if e]

etiquetas_edad = set(pd.concat([crudos[a]['Age'] for a in ANIOS]).dropna())
assert etiquetas_edad <= set(EDADES), f'Edades sin traducir: {etiquetas_edad - set(EDADES)}'


def estandarizar_edad(df):
    df = df.copy()
    df['age'] = pd.Categorical(df['age'].map(EDADES), categories=ORDEN_EDADES, ordered=True)
    return df

**País.** El diagnóstico encontró unos 30 países escritos distinto que en la fuente de PIB (OWID). Esos países reúnen cerca del 30 % de las respuestas. Se construye una tabla de equivalencias con los nombres de OWID. `Nomadic` no es un país: se convierte en nulo.

In [13]:
PAISES_EQUIVALENCIAS = {
    'United States of America': 'United States',
    'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
    'Russian Federation': 'Russia', 'Czech Republic': 'Czechia',
    'Iran, Islamic Republic of...': 'Iran', 'Viet Nam': 'Vietnam',
    'Hong Kong (S.A.R.)': 'Hong Kong', 'Venezuela, Bolivarian Republic of...': 'Venezuela',
    'Republic of Korea': 'South Korea', 'Republic of Moldova': 'Moldova',
    'Syrian Arab Republic': 'Syria', 'United Republic of Tanzania': 'Tanzania',
    'The former Yugoslav Republic of Macedonia': 'North Macedonia',
    'Republic of North Macedonia': 'North Macedonia',
    "Côte d'Ivoire": "Cote d'Ivoire", 'Libyan Arab Jamahiriya': 'Libya', 'Swaziland': 'Eswatini',
    'Congo, Republic of the...': 'Congo',
    'Democratic Republic of the Congo': 'Democratic Republic of Congo',
    "Democratic People's Republic of Korea": 'North Korea',
    "Lao People's Democratic Republic": 'Laos', 'Brunei Darussalam': 'Brunei',
    'Timor-Leste': 'East Timor', 'Micronesia, Federated States of...': 'Micronesia (country)',
    'Nomadic': None,
}


def estandarizar_pais(df):
    df = df.copy()
    df['country'] = df['country'].str.strip().replace(PAISES_EQUIVALENCIAS).astype('str')
    return df


paises_todos = pd.concat([crudos[a]['Country'] for a in ANIOS])
sin_pib = paises_todos.pipe(lambda s: estandarizar_pais(s.to_frame('country'))['country'])
sin_pib = sin_pib[sin_pib.notna() & ~sin_pib.isin(paises_owid)].value_counts()
print('Antes: respuestas con país sin coincidencia en OWID:', (~paises_todos.dropna().isin(paises_owid)).sum())
print('Después:', sin_pib.sum(), '\n')
sin_pib

Antes: respuestas con país sin coincidencia en OWID: 78152
Después: 1448 



country
Taiwan           858
Venezuela        341
Cuba              95
Yemen             70
Isle of Man       48
North Korea       25
Monaco             6
Liechtenstein      5
Name: count, dtype: int64

Los países que siguen sin coincidencia no están mal escritos: **OWID no publica su PIB** (Taiwán, Venezuela, Cuba, Yemen, Corea del Norte y algunos microestados). Se conservan, porque sus salarios son válidos; solo quedarán sin PIB en la Lección 4.

**Tecnologías.** `LanguageHaveWorkedWith` y `WebframeHaveWorkedWith` guardan varias opciones en una celda, separadas por `;`. Se buscan variantes de una misma tecnología comparando los nombres en minúsculas y sin signos, y se revisa en qué años aparece cada variante:

In [14]:
import re


def variantes(columna):
    """Agrupa etiquetas que solo difieren en mayúsculas, espacios, signos o el sufijo .js."""
    por_anio = pd.concat([crudos[a][columna].dropna().str.split(';').explode().to_frame('etiqueta').assign(year=a)
                          for a in ANIOS])
    por_anio['clave'] = por_anio['etiqueta'].str.lower().str.replace('.js', 'js').map(lambda t: re.sub(r'[^a-z0-9#+]', '', t))
    presencia = por_anio.groupby(['clave', 'etiqueta'])['year'].unique().map(sorted)
    repetidas = presencia.groupby(level='clave').size().loc[lambda s: s > 1].index
    return presencia.loc[repetidas].reset_index()


pd.concat([variantes('LanguageHaveWorkedWith').assign(columna='lenguajes'),
           variantes('WebframeHaveWorkedWith').assign(columna='frameworks')], ignore_index=True)

,clave,etiqueta,year,columna
0,cobol,COBOL,"[2022, 2025]",lenguajes
1,cobol,Cobol,"[2023, 2024]",lenguajes
2,lisp,LISP,[2022],lenguajes
3,lisp,Lisp,"[2023, 2024, 2025]",lenguajes
4,angularjs,Angular.js,[2022],frameworks
5,angularjs,AngularJS,"[2023, 2024, 2025]",frameworks
6,aspnetcore,ASP.NET CORE,"[2023, 2024]",frameworks
7,aspnetcore,ASP.NET Core,[2025],frameworks
8,aspnetcore,ASP.NET Core,[2022],frameworks


In [15]:
etiquetas_por_anio = {a: set(crudos[a]['LanguageHaveWorkedWith'].dropna().str.split(';').explode()) for a in ANIOS}
for etiqueta in ['Bash/Shell', 'Bash/Shell (all shells)']:
    print(f'{etiqueta!r}: {[a for a in ANIOS if etiqueta in etiquetas_por_anio[a]]}')

'Bash/Shell': [2022]
'Bash/Shell (all shells)': [2023, 2024, 2025]


Cada variante aparece en años distintos y nunca junto con su pareja: es la misma opción con el nombre cambiado. `Bash/Shell` (2022) pasó a llamarse `Bash/Shell (all shells)` (2023–2025). En cambio, **`Angular` y `AngularJS` no se unifican**, porque son frameworks distintos que la encuesta ofrece al mismo tiempo.

La función `estandarizar_lista` quita espacios, aplica las equivalencias, elimina repeticiones dentro de la celda y ordena las opciones. Se usa `map(..., na_action='ignore')` para no tocar los nulos.

In [16]:
EQUIV_LENGUAJES = {'Bash/Shell (all shells)': 'Bash/Shell', 'Cobol': 'COBOL', 'LISP': 'Lisp'}
EQUIV_FRAMEWORKS = {'ASP.NET CORE': 'ASP.NET Core', 'Angular.js': 'AngularJS', 'React.js': 'React'}


def estandarizar_lista(serie, equivalencias):
    """Limpia una columna de selección múltiple separada por ';'."""
    def limpiar(celda):
        opciones = {equivalencias.get(o.strip(), o.strip()) for o in celda.split(';')}
        return ';'.join(sorted(o for o in opciones if o))
    return serie.map(limpiar, na_action='ignore')


def estandarizar_tecnologias(df):
    df = df.copy()
    df['languages'] = estandarizar_lista(df['languages'], EQUIV_LENGUAJES)
    df['frameworks'] = estandarizar_lista(df['frameworks'], EQUIV_FRAMEWORKS)
    return df


ejemplo = pd.Series(['React.js;Node.js', 'ASP.NET Core ;jQuery', None])
estandarizar_lista(ejemplo, EQUIV_FRAMEWORKS)

0          Node.js;React
1    ASP.NET Core;jQuery
2                    NaN
dtype: str

### 2.5 Valores faltantes: comprobar la causa antes de decidir

Se aplican los pasos anteriores a los cuatro años para medir los nulos **dentro de la población de interés** (profesionales de tiempo completo), que es la que se analiza.

In [17]:
etapa = pd.concat([crudos[a].pipe(seleccionar_columnas, a).pipe(corregir_tipos).pipe(estandarizar_empleo)
                   for a in ANIOS], ignore_index=True)
en_poblacion = etapa['is_professional'] & etapa['full_time']

nulos_poblacion = (etapa[en_poblacion]
                   .groupby('year')[['salary_usd', 'languages', 'frameworks', 'country', 'age', 'work_exp', 'years_code_pro']]
                   .agg(lambda s: round(s.isna().mean() * 100, 1)))
nulos_poblacion

,salary_usd,languages,frameworks,country,age,work_exp,years_code_pro
year,,,,,,,
2022,26.60,0.40,20.90,0.00,0.80,28.10,5.10
2023,26.20,0.40,20.20,0.00,0.00,31.20,4.30
2024,52.80,8.00,27.20,9.70,0.00,38.90,8.00
2025,41.30,32.30,48.10,25.80,0.00,3.70,100.00


**Hipótesis 1: el salario falta sobre todo fuera de la población de interés**, porque la encuesta no lo pregunta a estudiantes ni a desempleados.

In [18]:
(etapa.assign(en_poblacion=en_poblacion)
      .groupby(['year', 'en_poblacion'])['salary_usd']
      .agg(lambda s: round(s.isna().mean() * 100, 1))
      .unstack()
      .rename(columns={False: '% nulo fuera', True: '% nulo dentro'}))

en_poblacion,% nulo fuera,% nulo dentro
year,,
2022,78.90,26.60
2023,76.40,26.20
2024,82.00,52.80
2025,65.40,41.30


Si se exige además el salario, que es la condición para entrar al análisis, los nulos de las otras columnas cambian:

In [19]:
(etapa[en_poblacion & etapa['salary_usd'].notna()]
      .groupby('year')[['languages', 'frameworks', 'country', 'work_exp']]
      .agg(lambda s: round(s.isna().mean() * 100, 1)))

,languages,frameworks,country,work_exp
year,,,,
2022,0.20,20.80,0.00,25.20
2023,0.20,19.70,0.00,27.80
2024,0.30,19.80,0.00,26.70
2025,6.50,27.60,0.00,0.90


**Hipótesis 2: en 2022–2024, a quien le falta `WorkExp` casi siempre tiene `YearsCodePro`**, y las dos variables miden casi lo mismo.

In [20]:
p = etapa[en_poblacion & (etapa['year'] < 2025)]
sin_workexp = p[p['work_exp'].isna()]
print('Sin WorkExp:', len(sin_workexp), '| de ellos con YearsCodePro (%):',
      round(sin_workexp['years_code_pro'].notna().mean() * 100, 1))

ambas = p.dropna(subset=['work_exp', 'years_code_pro'])
(ambas.groupby('year')
      .apply(lambda g: pd.Series({'casos con ambas': len(g),
                                  'correlación': g['work_exp'].astype(float).corr(g['years_code_pro'].astype(float)),
                                  'mediana |diferencia|': (g['work_exp'] - g['years_code_pro']).abs().median()}))
      .round(3))

Sin WorkExp: 44446 | de ellos con YearsCodePro (%): 90.2


,casos con ambas,correlación,mediana |diferencia|
year,,,
2022,"29,435.00",0.92,0.00
2023,"35,268.00",0.94,0.00
2024,"24,296.00",0.94,0.00


Las dos hipótesis se confirman:

- **Salario:** fuera de la población de interés falta en 65–82 % de las respuestas; dentro, en 26–53 %. La mayor parte de los nulos se debe al diseño de la encuesta.
- **Con salario, casi todo lo demás está completo:** el país no falta nunca (0 %) y los lenguajes faltan en menos de 0,3 % (6,5 % en 2025). Los nulos de país y lenguajes que el diagnóstico vio en 2024–2025 eran de personas sin salario.
- **Experiencia:** en 2022–2024 el 90,2 % de quienes no tienen `WorkExp` sí tienen `YearsCodePro`. Las dos variables tienen una correlación de 0,92 a 0,94 y la mediana de su diferencia es 0 años.

| Columna | Decisión | Justificación |
|---|---|---|
| `salary_usd` | **Eliminar** las filas sin salario (en la Lección 3) | Es la variable que se quiere explicar; imputarla inventaría el resultado |
| `work_exp` | **Corregir**: completar con `years_code_pro` en 2022–2024 | Miden casi lo mismo; la fuente queda registrada en `work_exp_source` |
| `languages` | **Conservar** el nulo | Se excluye solo de los indicadores por lenguaje; el salario de esa persona sigue sirviendo |
| `frameworks` | **Conservar** el nulo (20–28 % con salario) | Puede significar que no usa frameworks web (por ejemplo, desarrollo de sistemas o de datos); rellenarlo con "ninguno" sería suponer |
| `country` | **Conservar**; `Nomadic` → nulo | Sin país no hay región ni PIB, pero el resto de la respuesta es válido |
| `age` | `Prefer not to say` → nulo | No es una edad |

### 2.6 Atípicos e incoherencias

**Experiencia incoherente.** Se aplican tres reglas. Primero, `years_code_pro` mayor que `years_code` (más años programando profesionalmente que programando) es imposible, así que se anula antes de usarla para completar. Después de completar, también se anula `work_exp` si supera los 50 años o si supera la edad máxima del rango declarado menos 14 años. Los valores se convierten en nulos, no se borra la fila, porque el salario y las tecnologías de esa persona siguen siendo válidos.

In [21]:
EDAD_MAXIMA = {'Under 18': 17, '18-24': 24, '25-34': 34, '35-44': 44, '45-54': 54, '55-64': 64, '65+': np.inf}
EDAD_INICIO_TRABAJO = 14


def corregir_experiencia(df):
    """Anula experiencia imposible, completa WorkExp con YearsCodePro y registra la fuente."""
    df = df.copy()
    pro_mayor = (df['years_code_pro'] > df['years_code']).fillna(False)
    df.loc[pro_mayor, 'years_code_pro'] = pd.NA

    tenia_workexp = df['work_exp'].notna()
    df['work_exp'] = df['work_exp'].fillna(df['years_code_pro'])
    fuente = np.where(tenia_workexp, 'WorkExp', np.where(df['work_exp'].notna(), 'YearsCodePro', None))
    df['work_exp_source'] = pd.Categorical(fuente, categories=['WorkExp', 'YearsCodePro'])

    limite = df['age'].map(EDAD_MAXIMA).astype(float) - EDAD_INICIO_TRABAJO
    imposible = ((df['work_exp'] > 50) | (df['work_exp'] > limite)).fillna(False)
    df.loc[imposible, ['work_exp', 'work_exp_source']] = pd.NA
    return df.drop(columns=['years_code', 'years_code_pro'])

`years_code` y `years_code_pro` se eliminan al final porque su información ya queda en `work_exp`, la variable común a los cuatro años. La columna `work_exp_source` registra de dónde salió cada valor.

**Salarios atípicos.** El diagnóstico propuso recortar fuera de los percentiles 1 y 99 de cada año. Se calculan dentro de la población de interés. Además se fijan un **piso de 1.000 USD** y un **tope de 1.000.000 USD** al año, por si un percentil queda en un valor imposible para un trabajo de tiempo completo.

In [22]:
PISO_SALARIO = 1_000
TOPE_SALARIO = 1_000_000


def calcular_limites_salario(df):
    """Límites de salario plausible por año: P1–P99 de la población, con piso y tope absolutos."""
    base = df[df['is_professional'] & df['full_time'] & df['salary_usd'].notna()]
    limites = base.groupby('year')['salary_usd'].quantile([0.01, 0.99]).unstack()
    limites.columns = ['p01', 'p99']
    limites['limite_inferior'] = limites['p01'].clip(lower=PISO_SALARIO)
    limites['limite_superior'] = limites['p99'].clip(upper=TOPE_SALARIO)
    return limites


calcular_limites_salario(etapa)

,p01,p99,limite_inferior,limite_superior
year,,,,
2022,"1,548.00","2,237,385.00","1,548.00","1,000,000.00"
2023,495.88,"400,000.00","1,000.00","400,000.00"
2024,282.84,"400,000.00","1,000.00","400,000.00"
2025,125.90,"441,215.30","1,000.00","441,215.30"


- **Tope:** el percentil 99 de 2022 es 2,24 millones de USD, un valor imposible para la mayoría de los desarrolladores. El tope de 1 millón solo actúa en 2022; en 2023–2025 el percentil 99 ya está entre 400.000 y 441.000 USD.
- **Piso:** el percentil 1 de 2023–2025 está entre 126 y 496 USD al año, otro valor imposible para un trabajo de tiempo completo. Lo más probable es un salario mensual escrito como anual o una moneda mal elegida. Por eso actúa el piso de 1.000 USD. En 2022 el percentil 1 (1.548 USD) queda como límite.

**Decisión: eliminar** las respuestas fuera de estos límites. El filtro se aplica en la Lección 3 con `between`.

### 2.7 Funciones reutilizables y pipeline con `pipe()`

Cada función de las secciones anteriores recibe un DataFrame y devuelve otro, así que se pueden encadenar con `pipe()`. Entre paso y paso, `anotar` toma una "foto" de la tabla (filas, columnas, nulos y memoria) y la devuelve **sin cambios**. Así el pipeline deja un informe de lo que cambió en cada paso.

In [23]:
registro = []


def anotar(df, tabla, paso):
    registro.append({'tabla': tabla, 'paso': paso, 'filas': len(df), 'columnas': df.shape[1],
                     'nulos': int(df.isna().sum().sum()),
                     'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1)})
    return df


def limpiar_encuesta(crudo, anio):
    tabla = f'so_{anio}'
    return (crudo
            .pipe(anotar, tabla, '0 crudo')
            .pipe(seleccionar_columnas, anio)
            .pipe(corregir_tipos)
            .pipe(anotar, tabla, '1 columnas y tipos')
            .pipe(estandarizar_empleo)
            .pipe(estandarizar_edad)
            .pipe(estandarizar_pais)
            .pipe(anotar, tabla, '2 empleo, edad y país')
            .pipe(estandarizar_tecnologias)
            .pipe(anotar, tabla, '3 tecnologías')
            .pipe(corregir_experiencia)
            .pipe(anotar, tabla, '4 experiencia'))


limpias = {anio: limpiar_encuesta(crudos[anio], anio) for anio in ANIOS}
limpias[2023].head(3)

,year,response_id,country,age,work_exp,languages,frameworks,salary_usd,is_professional,full_time,work_exp_source
0,2023,1,NaN,18-24,<NA>,NaN,NaN,<NA>,False,False,NaN
1,2023,2,United States,25-34,10.00,HTML/CSS;JavaScript;Python,Next.js;React;Remix;Vue.js,"285,000.00",True,True,WorkExp
2,2023,3,United States,45-54,23.00,Bash/Shell;Go,NaN,"250,000.00",True,True,WorkExp


Las dos fuentes de PIB también pasan por un pipeline. En OWID se quitan los 13 agregados (`World`, regiones y niveles de ingreso), que se reconocen porque no tienen `owid_region`. En el Banco Mundial se eliminan la columna vacía y las dos columnas constantes, y se conservan solo los códigos ISO3 que OWID reconoce como países. El archivo del Banco Mundial no trae una columna que marque los agregados.

In [24]:
def limpiar_owid(df):
    return (df[df['owid_region'].notna()]
            .rename(columns={'entity': 'country', 'code': 'iso3',
                             'ny_gdp_pcap_pp_kd': 'gdp_pc_ppp_owid', 'owid_region': 'region'})
            .astype({'year': 'int16'})
            .reset_index(drop=True))


def solo_paises(df, codigos):
    return df[df['iso3'].isin(codigos)].reset_index(drop=True)


owid = (owid_crudo
        .pipe(anotar, 'gdp_owid', '0 crudo')
        .pipe(limpiar_owid)
        .pipe(anotar, 'gdp_owid', '1 sin agregados'))

bm = (bm_crudo
      .pipe(anotar, 'gdp_wb', '0 crudo')
      .drop(columns=['Indicator Name', 'Indicator Code', 'Unnamed: 70'])
      .rename(columns={'Country Name': 'country_wb', 'Country Code': 'iso3'})
      .pipe(anotar, 'gdp_wb', '1 sin columnas vacías o constantes')
      .pipe(solo_paises, set(owid['iso3']))
      .pipe(anotar, 'gdp_wb', '2 solo países'))

In [25]:
informe = pd.DataFrame(registro)
informe

,tabla,paso,filas,columnas,nulos,memoria_mb
0,so_2022,0 crudo,73268,11,122463,17.10
1,so_2022,1 columnas y tipos,73268,12,122463,17.00
2,so_2022,"2 empleo, edad y país",73268,12,121462,9.60
3,so_2022,3 tecnologías,73268,12,121462,9.50
4,so_2022,4 experiencia,73268,11,100421,8.20
5,so_2023,0 crudo,89184,11,138354,21.80
6,so_2023,1 columnas y tipos,89184,12,138354,21.70
7,so_2023,"2 empleo, edad y país",89184,12,137578,12.40
8,so_2023,3 tecnologías,89184,12,137578,12.00
9,so_2023,4 experiencia,89184,11,109570,10.50


- **Ningún paso de las encuestas cambia el número de filas.** La limpieza corrige valores y deja la selección de filas para la Lección 3.
- **La memoria se reduce a la mitad:** 2022 pasa de 17,1 a 8,2 MB, 2023 de 21,8 a 10,5, 2024 de 15,7 a 7,4 y 2025 de 9,9 a 4,9. El mayor ahorro está en el paso 2, porque los textos largos de `MainBranch` y `Employment` se reemplazan por dos columnas booleanas y la edad pasa a categoría.
- **Los nulos bajan en el paso 4**, porque se completa `work_exp` y se eliminan las columnas que ya no se necesitan. En 2025 suben en 49.191 en el paso 1, porque se agrega `years_code_pro` vacía para tener las mismas columnas que los otros años; esa columna sale en el paso 4. En 2024 suben un poco en el paso 2: `Prefer not to say` y `Nomadic` pasan a ser nulos reales.
- **PIB:** OWID pierde 468 filas, que son los 13 agregados, y con ellas todos sus nulos. El Banco Mundial pasa de 266 entidades a 198 países y de 71 a 68 columnas.

### 2.8 Validación con `assert`

Las reglas que deben cumplir los datos limpios se convierten en una función. Si alguna falla, el cuaderno se detiene y muestra qué regla no se cumplió.

In [26]:
def validar_encuesta(df, crudo):
    assert len(df) == len(crudo), 'la limpieza no debe eliminar filas'
    assert list(df.columns) == list(limpias[ANIOS[0]].columns), 'columnas distintas entre años'
    assert df['response_id'].is_unique, 'response_id repetido'
    assert df['salary_usd'].notna().sum() == crudo['ConvertedCompYearly'].notna().sum(), 'cambió el número de salarios'
    assert df['salary_usd'].dropna().gt(0).all(), 'salario no positivo'
    assert df['work_exp'].dropna().between(0, 50).all(), 'experiencia fuera de rango'
    assert df['work_exp'].isna().eq(df['work_exp_source'].isna()).all(), 'experiencia sin fuente'
    assert not df['country'].isin(list(PAISES_EQUIVALENCIAS)).any(), 'país sin estandarizar'
    for col, equivalencias in [('languages', EQUIV_LENGUAJES), ('frameworks', EQUIV_FRAMEWORKS)]:
        etiquetas = set(df[col].dropna().str.split(';').explode())
        assert not etiquetas & set(equivalencias), f'{col}: quedan variantes sin unificar'
        assert all(e == e.strip() and e for e in etiquetas), f'{col}: etiquetas con espacios o vacías'
    return 'ok'


def validar_pib(owid, bm):
    assert not owid.duplicated(['iso3', 'year']).any(), 'OWID: país-año repetido'
    assert owid['iso3'].notna().all() and owid['region'].notna().all(), 'OWID: país sin código o región'
    assert owid['gdp_pc_ppp_owid'].gt(0).all(), 'OWID: PIB no positivo'
    assert bm['iso3'].is_unique, 'BM: país repetido'
    return 'ok'


print({anio: validar_encuesta(limpias[anio], crudos[anio]) for anio in ANIOS}, '| PIB:', validar_pib(owid, bm))

{2022: 'ok', 2023: 'ok', 2024: 'ok', 2025: 'ok'} | PIB: ok


### 2.9 Unión de los cuatro años limpios

Con las mismas columnas y tipos, los cuatro años se apilan con `concat` para analizarlos juntos en la Lección 3. La validación detallada de esta unión se hace en la Lección 4.

In [27]:
encuestas = pd.concat(limpias.values(), ignore_index=True)
print(encuestas.shape)
encuestas.dtypes

(277080, 11)


year                  int16
response_id           int32
country                 str
age                category
work_exp            Float64
languages               str
frameworks              str
salary_usd          Float64
is_professional        bool
full_time              bool
work_exp_source    category
dtype: object

**Resumen de la Lección 2: problema del diagnóstico → tratamiento**

| Problema | Tratamiento | Resultado |
|---|---|---|
| Duplicados | Revisión exacta e implícita | 0 exactos; 15 parciales conservados porque no son iguales en la encuesta completa |
| Columnas vacías y preguntas que no se usan | `usecols` con 11 columnas | Las columnas `VCHosting*` no se cargan |
| Experiencia como texto | `replace` + `to_numeric` → `Float64` | 0 valores perdidos en la conversión |
| `YearsCodePro` no existe en 2025 | `work_exp` como variable común, completada con `years_code_pro` | Variable comparable en los cuatro años |
| Experiencia incoherente | Reglas de edad, máximo de 50 años y `years_code_pro` ≤ `years_code` | Valores imposibles anulados, sin borrar filas |
| `Employment` y `MainBranch` cambian entre años | Dos columnas booleanas: `is_professional` y `full_time` | Un mismo criterio para filtrar los cuatro años |
| Nombres de país distintos de OWID | Tabla de 25 equivalencias | Respuestas sin coincidencia: de 78.152 a 1.448, solo de países sin dato de PIB |
| Variantes de tecnologías | `strip` + equivalencias | Lenguajes: de 58 a 55 etiquetas; frameworks: de 42 a 39 |
| Agregados en las fuentes de PIB | Filtro por `owid_region` y por código ISO3 | Solo quedan países |

Cada paso está en una función y todas se encadenan con `pipe()`. `validar_encuesta` y `validar_pib` comprueban las reglas con `assert` cada vez que se ejecuta el cuaderno.

## Lección 3: Filtrado, ordenamiento y agrupación de datos

Cada pregunta se traduce primero a la forma "métrica por grupo". Por ejemplo, "¿qué lenguajes pagan más?" se convierte en "salario mediano relativo **por** lenguaje". Se usa la **mediana** y no la media porque el salario tiene una cola larga hacia arriba, incluso después del recorte.

### 3.1 Población de interés con filtros booleanos

In [28]:
limites = calcular_limites_salario(encuestas)

es_profesional = encuestas['is_professional']
tiempo_completo = encuestas['full_time']
con_salario = encuestas['salary_usd'].notna()
salario_plausible = encuestas['salary_usd'].between(encuestas['year'].map(limites['limite_inferior']),
                                                    encuestas['year'].map(limites['limite_superior'])).fillna(False)

embudo = pd.DataFrame({
    '1 todas las respuestas': encuestas.groupby('year').size(),
    '2 profesionales': encuestas[es_profesional].groupby('year').size(),
    '3 de tiempo completo': encuestas[es_profesional & tiempo_completo].groupby('year').size(),
    '4 con salario': encuestas[es_profesional & tiempo_completo & con_salario].groupby('year').size(),
    '5 con salario plausible': encuestas[es_profesional & tiempo_completo & salario_plausible].groupby('year').size(),
})
embudo.loc['total'] = embudo.sum()
embudo

,1 todas las respuestas,2 profesionales,3 de tiempo completo,4 con salario,5 con salario plausible
year,,,,,
2022,73268,53507,43199,31716,30686
2023,89184,67237,53663,39623,38734
2024,65437,50207,39937,18843,18286
2025,49191,37467,28774,16891,16390
total,277080,208418,165573,107073,104096


In [29]:
poblacion = (encuestas[es_profesional & tiempo_completo & salario_plausible]
             .drop(columns=['is_professional', 'full_time'])
             .reset_index(drop=True))
print(poblacion.shape)

(104096, 9)


Con `query` se separa la **población estricta** de 2022–2024, donde "tiempo completo" se mide de forma exacta, y con `isin` se cuentan los países con más respuestas:

In [30]:
estricta = poblacion.query('year <= 2024')
print('Población estricta 2022–2024:', len(estricta), '| 2025 (aproximada):', len(poblacion) - len(estricta))

top_paises = poblacion['country'].value_counts().head(5)
print('Los 5 países con más respuestas reúnen el',
      round(poblacion['country'].isin(top_paises.index).mean() * 100, 1), '% de la población')
top_paises

Población estricta 2022–2024: 87706 | 2025 (aproximada): 16390
Los 5 países con más respuestas reúnen el 48.2 % de la población


country
United States     24824
Germany            8343
United Kingdom     7382
India              5276
Canada             4361
Name: count, dtype: int64

### 3.2 Categorías: región y nivel de experiencia

**Región.** Se toma la región continental de OWID (`owid_region`) y se lleva a cada respuesta con `map`. A los países que OWID no incluye se les asigna la región a mano. Con unas 70–100 naciones de menos de 30 respuestas por año, el diagnóstico recomendó analizar por región y no por país.

**Nivel de experiencia.** Con `pd.cut` se convierten los años de experiencia en cinco tramos.

In [31]:
REGION_MANUAL = {'Taiwan': 'Asia', 'Venezuela': 'South America', 'Cuba': 'North America', 'Yemen': 'Asia',
                 'North Korea': 'Asia', 'Isle of Man': 'Europe', 'Monaco': 'Europe', 'Liechtenstein': 'Europe'}
region_por_pais = pd.concat([owid.drop_duplicates('country').set_index('country')['region'],
                             pd.Series(REGION_MANUAL)])
assert region_por_pais.index.is_unique

NIVELES = ['0-2', '3-5', '6-10', '11-20', '21+']
poblacion['region'] = poblacion['country'].map(region_por_pais).astype('category')
poblacion['exp_level'] = pd.cut(poblacion['work_exp'], bins=[-np.inf, 2, 5, 10, 20, np.inf], labels=NIVELES)

print('Sin región:', poblacion['region'].isna().sum(), '| sin nivel de experiencia:', poblacion['exp_level'].isna().sum())
poblacion['region'].value_counts()

Sin región:

 23 | sin nivel de experiencia: 574


region
Europe           49422
North America    30484
Asia             13880
South America     5145
Oceania           3161
Africa            1981
Name: count, dtype: int64

### 3.3 Salario relativo con `transform`

Los salarios cambian mucho entre años y regiones. Si se compararan en bruto, una tecnología muy usada en Estados Unidos parecería mejor pagada solo por el país. Por eso, con `groupby().transform('median')`, se divide cada salario entre la mediana de su grupo:

- `salary_rel_year`: salario / mediana del mismo año. Mide la **diferencia bruta**.
- `salary_rel_region`: salario / mediana del mismo año **y región**. Compara a cada persona con quienes viven en su misma región.
- `salary_rel_group`: salario / mediana del mismo año, región **y nivel de experiencia**. Se usa en el indicador 5.

Un valor de 1,20 significa que la persona gana 20 % más que la mediana de su grupo. Las personas sin país (`Nomadic`) no tienen región y quedan sin salario relativo regional.

In [32]:
def relativo(df, grupo):
    """Salario dividido entre la mediana de su grupo."""
    return df['salary_usd'] / df.groupby(grupo, observed=True)['salary_usd'].transform('median')


poblacion['salary_rel_year'] = relativo(poblacion, 'year')
poblacion['salary_rel_region'] = relativo(poblacion, ['year', 'region'])
poblacion['salary_rel_group'] = relativo(poblacion, ['year', 'region', 'exp_level'])
poblacion[['year', 'country', 'region', 'exp_level', 'salary_usd', 'salary_rel_year', 'salary_rel_region']].head()

,year,country,region,exp_level,salary_usd,salary_rel_year,salary_rel_region
0,2022,Israel,Asia,11-20,"215,232.00",3.21,9.27
1,2022,Netherlands,Europe,6-10,"49,056.00",0.73,0.82
2,2022,United Kingdom,Europe,0-2,"60,307.00",0.90,1.01
3,2022,United States,North America,3-5,"65,000.00",0.97,0.50
4,2022,United States,North America,3-5,"110,000.00",1.64,0.85


### 3.4 Tecnologías en formato largo

Para agrupar por tecnología, cada celda con varias opciones se separa con `str.split(';')` y `explode()`: una fila por persona y tecnología. Se conservan las columnas necesarias para los indicadores.

In [33]:
ATRIBUTOS = ['year', 'response_id', 'region', 'exp_level', 'salary_usd',
             'salary_rel_year', 'salary_rel_region', 'salary_rel_group']


def a_formato_largo(df, columna, nombre):
    largo = (df[ATRIBUTOS + [columna]]
             .dropna(subset=[columna])
             .assign(**{nombre: lambda d: d[columna].str.split(';')})
             .explode(nombre)
             .drop(columns=columna)
             .reset_index(drop=True))
    esperado = df[columna].dropna().str.count(';').add(1).sum()
    assert len(largo) == esperado, 'explode no generó una fila por opción'
    return largo


uso_lenguajes = a_formato_largo(poblacion, 'languages', 'language')
uso_frameworks = a_formato_largo(poblacion, 'frameworks', 'framework')
print('lenguajes:', uso_lenguajes.shape, '| frameworks:', uso_frameworks.shape)

lenguajes: (540902, 9) | frameworks: (282657, 9)


### 3.5 Indicador 1: prima salarial por lenguaje

**Métrica por grupo:** mediana del salario relativo por lenguaje. La **prima** es esa mediana menos 1, en porcentaje. Para que la cifra sea confiable se exige un mínimo de **300 respuestas** y que el lenguaje aparezca en los cuatro años.

In [34]:
MIN_RESPUESTAS = 300


def resumen_tecnologia(largo, columna, total_personas):
    resumen = (largo.groupby(columna)
               .agg(respuestas=('response_id', 'size'),
                    anios=('year', 'nunique'),
                    salario_mediano=('salary_usd', 'median'),
                    rel_anio=('salary_rel_year', 'median'),
                    rel_region=('salary_rel_region', 'median')))
    resumen['uso_pct'] = resumen['respuestas'] / total_personas * 100
    resumen['prima_bruta_pct'] = (resumen['rel_anio'] - 1) * 100
    resumen['prima_region_pct'] = (resumen['rel_region'] - 1) * 100
    resumen = (resumen.drop(columns=['rel_anio', 'rel_region'])
                      .query('respuestas >= @MIN_RESPUESTAS and anios == 4')
                      .sort_values('prima_region_pct', ascending=False))
    pct = ['uso_pct', 'prima_bruta_pct', 'prima_region_pct']
    resumen[pct] = resumen[pct].round(1) + 0.0          # + 0.0 evita que se muestre "-0.0"
    return resumen


ind_lenguajes = resumen_tecnologia(uso_lenguajes, 'language', poblacion['languages'].notna().sum())
print('Lenguajes con datos suficientes:', len(ind_lenguajes))
ind_lenguajes.head(10)

Lenguajes con datos suficientes: 34


,respuestas,anios,salario_mediano,uso_pct,prima_bruta_pct,prima_region_pct
language,,,,,,
Scala,3264,4,"94,230.00",3.20,30.20,28.60
Erlang,1106,4,"101,925.50",1.10,39.40,28.30
Elixir,2740,4,"98,814.00",2.70,35.00,25.00
Ruby,7363,4,"98,000.00",7.20,34.60,17.90
Go,14992,4,"89,379.00",14.60,21.80,16.90
F#,1073,4,"93,244.00",1.00,27.80,15.40
Groovy,4638,4,"86,085.00",4.50,18.90,15.40
OCaml,474,4,"91,173.50",0.50,25.10,14.30
Lisp,1145,4,"97,046.00",1.10,32.90,12.20


In [35]:
ind_lenguajes.nsmallest(5, 'prima_region_pct')

,respuestas,anios,salario_mediano,uso_pct,prima_bruta_pct,prima_region_pct
language,,,,,,
PHP,17947,4,"55,862.00",17.40,-23.00,-16.70
VBA,3110,4,"64,509.00",3.00,-9.30,-13.90
Dart,5318,4,"51,403.00",5.20,-29.10,-12.60
MATLAB,2359,4,"63,984.00",2.30,-12.80,-10.70
COBOL,501,4,"75,000.00",0.50,3.10,-7.70


**Indicador 1. Prima salarial por lenguaje.** De 34 lenguajes con datos suficientes:

- **Primas más altas dentro de la región:** Scala (+28,6 %), Erlang (+28,3 %), Elixir (+25,0 %), Ruby (+17,9 %) y Go (+16,9 %). Les siguen F#, Groovy, OCaml, Lisp y Rust (+12,2 %). Son sobre todo lenguajes de *backend* y sistemas distribuidos. Go y Rust combinan una prima alta con un uso amplio (14,6 % y 11,0 % de la población); los demás son de nicho, con 0,5 % a 7,2 %.
- **Primas más bajas:** PHP (−16,7 %, usado por el 17,4 %), VBA (−13,9 %), Dart (−12,6 %), MATLAB (−10,7 %) y COBOL (−7,7 %).
- **Comparar dentro de la región reduce casi todas las primas.** Ruby pasa de +34,6 % (bruta) a +17,9 %, Lisp de +32,9 % a +12,2 % y Dart de −29,1 % a −12,6 %. Parte de la diferencia bruta se debe a **dónde viven** quienes usan cada lenguaje, no al lenguaje. Aun así, la mayoría conserva el signo: la asociación no desaparece al comparar dentro de la región.

**Evolución en el tiempo.** Con `pivot_table` se calcula la prima bruta de los 10 lenguajes más usados en cada año:

In [36]:
mas_usados = ind_lenguajes.nlargest(10, 'uso_pct').index
prima_anual_lenguajes = (uso_lenguajes[uso_lenguajes['language'].isin(mas_usados)]
                         .pivot_table(index='language', columns='year', values='salary_rel_year', aggfunc='median')
                         .sub(1).mul(100).round(1).add(0.0)
                         .sort_values(2025, ascending=False))
prima_anual_lenguajes

year,2022,2023,2024,2025
language,,,,
Bash/Shell,21.20,14.90,14.00,9.00
Python,6.50,7.10,3.90,2.20
C++,3.30,2.60,0.00,1.80
C#,2.90,0.00,0.00,1.10
TypeScript,3.30,4.00,0.00,1.10
SQL,1.00,0.00,-1.70,0.80
JavaScript,-3.20,-0.70,-5.40,-0.40
Java,-4.70,-2.90,-7.70,-1.80
HTML/CSS,-4.70,-4.00,-7.40,-2.60


**Indicador 1. Prima salarial por lenguaje.** De 34 lenguajes con datos suficientes:

- **Primas más altas dentro de la región:** Scala (+28,6 %), Erlang (+28,3 %), Elixir (+25,0 %), Ruby (+17,9 %) y Go (+16,9 %). Les siguen F#, Groovy, OCaml, Lisp y Rust (+12,2 %). Son sobre todo lenguajes de *backend* y sistemas distribuidos. Go y Rust combinan una prima alta con un uso amplio (14,6 % y 11,0 % de la población); los demás son de nicho, con 0,5 % a 7,2 %.
- **Primas más bajas:** PHP (−16,7 %, usado por el 17,4 %), VBA (−13,9 %), Dart (−12,6 %), MATLAB (−10,7 %) y COBOL (−7,7 %).
- **Comparar dentro de la región reduce casi todas las primas.** Ruby pasa de +34,6 % (bruta) a +17,9 %, Lisp de +32,9 % a +12,2 % y Dart de −29,1 % a −12,6 %. Parte de la diferencia bruta se debe a **dónde viven** quienes usan cada lenguaje, no al lenguaje. Aun así, la mayoría conserva el signo: la asociación no desaparece al comparar dentro de la región.B

### 3.6 Indicador 2: prima salarial por framework

In [37]:
ind_frameworks = resumen_tecnologia(uso_frameworks, 'framework', poblacion['frameworks'].notna().sum())
print('Frameworks con datos suficientes:', len(ind_frameworks))
ind_frameworks

Frameworks con datos suficientes: 23


,respuestas,anios,salario_mediano,uso_pct,prima_bruta_pct,prima_region_pct
framework,,,,,,
Phoenix,2059,4,"99,383.00",2.50,37.20,26.90
Ruby on Rails,5537,4,"95,541.00",6.80,31.40,16.50
FastAPI,7393,4,"73,516.00",9.00,0.00,7.10
Fastify,1865,4,"69,608.00",2.30,-5.00,6.70
Deno,1608,4,"80,000.00",2.00,6.90,3.80
Flask,9685,4,"71,683.00",11.80,-0.70,2.20
React,37310,4,"74,490.50",45.50,1.70,0.70
Svelte,4369,4,"75,410.00",5.30,3.40,0.00
Next.js,13841,4,"67,186.00",16.90,-7.20,0.00


**Indicador 2. Prima salarial por framework.** Los frameworks repiten el patrón de su lenguaje:

- **Más altas:** Phoenix (+26,9 %, el framework de Elixir), Ruby on Rails (+16,5 %), FastAPI (+7,1 %) y Fastify (+6,7 %).
- **Más bajas:** Laravel (−23,3 %) y Symfony (−13,4 %), los dos de PHP, además de jQuery (−12,8 %), Drupal (−10,7 %) y Nuxt.js (−10,1 %).
- **Los más usados quedan cerca de cero.** React (45,5 % de uso) tiene +0,7 % y Node.js −2,0 %. Cuando casi todos usan una tecnología, no puede marcar una diferencia de salario.

### 3.7 Indicador 3: salario por nivel de experiencia

**Métrica por grupo:** salario mediano por nivel de experiencia y año. El **multiplicador** compara el tramo de más de 20 años con el de 0–2 años.

In [38]:
salario_experiencia = poblacion.pivot_table(index='exp_level', columns='year', values='salary_usd',
                                            aggfunc='median', observed=True)
salario_experiencia.index = salario_experiencia.index.astype(str)   # para poder agregar una fila de texto
salario_experiencia.loc['multiplicador 21+ / 0-2'] = salario_experiencia.loc['21+'] / salario_experiencia.loc['0-2']
salario_experiencia.round(1)

year,2022,2023,2024,2025
exp_level,,,,
0-2,"31,993.00","40,000.00","32,170.00","31,993.00"
3-5,"49,020.00","56,280.00","47,259.00","47,334.00"
6-10,"70,380.00","77,491.50","70,063.00","75,410.00"
11-20,"87,448.00","92,545.00","89,087.00","97,514.00"
21+,"110,124.00","110,000.00","107,601.50","114,616.00"
multiplicador 21+ / 0-2,3.40,2.80,3.30,3.60


In [39]:
(poblacion.groupby('exp_level', observed=True)
          .agg(respuestas=('response_id', 'size'),
               salario_mediano=('salary_usd', 'median'),
               rel_region_mediano=('salary_rel_region', 'median'))
          .round(2))

,respuestas,salario_mediano,rel_region_mediano
exp_level,,,
0-2,12278,"35,340.00",0.56
3-5,21757,"51,402.00",0.78
6-10,28446,"74,597.00",1.06
11-20,26925,"91,295.00",1.21
21+,14116,"110,202.50",1.23


**Indicador 3. Salario por experiencia.** El salario mediano sube en cada tramo, de unos 32.000–40.000 USD con 0–2 años a unos 108.000–115.000 USD con más de 20 años: un **multiplicador de 2,8 a 3,6** según el año. El salto mayor está entre 3–5 y 6–10 años (de 51.402 a 74.597 USD en el total), y después de 20 años casi se detiene: 1,21 y 1,23 veces la mediana de su región.

La experiencia pesa mucho más que cualquier tecnología: pasar de 0–2 a 6–10 años casi duplica el salario relativo (0,56 → 1,06), mientras que la mayor prima de un lenguaje es cercana a +29 %.

### 3.8 Indicador 4: salario por región

In [40]:
salario_region = (poblacion.groupby('region', observed=True)
                  .agg(respuestas=('response_id', 'size'),
                       paises=('country', 'nunique'),
                       salario_mediano=('salary_usd', 'median'))
                  .sort_values('salario_mediano', ascending=False))
salario_region['participacion_pct'] = (salario_region['respuestas'] / len(poblacion) * 100).round(1)
salario_region

,respuestas,paises,salario_mediano,participacion_pct
region,,,,
North America,30484,20,"135,000.00",29.30
Oceania,3161,5,"91,013.00",3.00
Europe,49422,44,"64,254.00",47.50
South America,5145,12,"29,214.00",4.90
Asia,13880,49,"24,600.00",13.30
Africa,1981,44,"20,447.00",1.90


In [41]:
poblacion.pivot_table(index='region', columns='year', values='salary_usd', aggfunc='median', observed=True).round(0)

year,2022,2023,2024,2025
region,,,,
Africa,"19,938.00","21,718.00","18,000.00","20,883.00"
Asia,"23,215.00","25,998.00","24,707.00","26,735.00"
Europe,"59,720.00","64,254.00","62,000.00","72,393.00"
North America,"130,000.00","140,000.00","130,000.00","140,000.00"
Oceania,"88,464.00","89,540.00","90,840.00","97,514.00"
South America,"27,420.00","30,431.00","27,078.00","32,767.00"


**Indicador 4. Salario por región.** La mediana de América del Norte (135.000 USD) es **6,6 veces** la de África (20.447 USD). Europa (64.254 USD) aporta casi la mitad de las respuestas (47,5 %). El orden es el mismo los cuatro años. Europa es la región que más sube: de 59.720 USD en 2022 a 72.393 USD en 2025.

Hay que tener en cuenta que cada región mezcla países muy distintos. "Asia" junta a Israel y Japón con India, y "North America" incluye a México y Centroamérica, aunque la domina Estados Unidos. Por eso las comparaciones de tecnologías se hacen **dentro de cada región** con el salario relativo.

### 3.9 Indicador 5: ¿cambia la prima según la experiencia y la región?

Esta es la segunda parte de la pregunta analítica. Para cada lenguaje se calcula la prima **dentro de cada región** y **dentro de cada nivel de experiencia**, comparando con la mediana del mismo año, región y nivel. Solo se muestran las celdas con al menos 100 respuestas; las demás quedan vacías (`NaN`).

In [42]:
MIN_CELDA = 100
seleccion = ind_lenguajes.nlargest(12, 'uso_pct').index
base = uso_lenguajes[uso_lenguajes['language'].isin(seleccion)]


def prima_cruzada(columna):
    tabla = base.pivot_table(index='language', columns=columna, values='salary_rel_group',
                             aggfunc=['median', 'size'], observed=True)
    prima = (tabla['median'] - 1) * 100
    return prima.where(tabla['size'] >= MIN_CELDA).round(1) + 0.0


prima_por_experiencia = prima_cruzada('exp_level')
prima_por_experiencia.sort_values('21+', ascending=False)

exp_level,0-2,3-5,6-10,11-20,21+
language,,,,,
Go,19.90,16.80,16.70,17.30,16.70
Python,6.20,5.00,6.00,4.20,6.20
Bash/Shell,11.30,7.50,7.10,5.20,5.30
TypeScript,2.10,1.80,2.00,3.00,5.00
Java,0.00,0.00,0.70,2.30,3.30
C++,8.30,1.60,0.00,0.00,3.20
JavaScript,-3.80,-4.00,-2.60,-2.30,0.00
PowerShell,3.50,-4.50,-5.60,-4.10,-0.50
SQL,-1.30,-3.90,-3.10,-3.60,-2.00


In [43]:
prima_por_region = prima_cruzada('region')
prima_por_region.sort_values('Europe', ascending=False)

region,Africa,Asia,Europe,North America,Oceania,South America
language,,,,,,
Go,26.90,42.90,12.60,17.90,12.50,20.00
Bash/Shell,22.10,34.80,4.50,6.20,3.70,8.80
Python,3.60,19.10,4.00,5.80,2.10,3.20
TypeScript,6.60,6.50,2.00,2.70,3.60,7.70
Java,-6.90,2.70,0.00,4.70,5.40,0.00
C++,3.10,14.20,0.00,2.60,-1.50,-4.70
PowerShell,10.50,0.00,-0.60,-6.90,-2.20,-6.00
JavaScript,-4.90,-7.00,-2.60,-2.10,0.00,-2.40
SQL,-0.20,-5.40,-2.80,-3.30,0.00,-4.00


**Indicador 5. La prima cambia según la experiencia y la región.**

- **Por experiencia:** Go mantiene una prima de +17 % a +20 % en todos los tramos, así que no se explica por la experiencia de quienes lo usan. Bash/Shell y C++ pagan más en los primeros años (+11,3 % y +8,3 % con 0–2 años) y la prima baja después. TypeScript y Java hacen lo contrario: su prima crece con los años (de +2,1 % a +5,0 % y de 0 % a +3,3 %). La penalización de PHP se reduce con la experiencia (de −19 % a −12 %).
- **Por región:** Go tiene prima positiva en las seis regiones (de +12,5 % en Oceanía a +42,9 % en Asia) y PHP la tiene negativa en todas (de −7,1 % a −37,8 %). Las primas son mucho mayores en **Asia y África**, donde los salarios son más desiguales. Allí usar Go, Bash/Shell o Python puede indicar trabajo en empresas extranjeras o remotas. En **Europa y Oceanía** las primas son pequeñas: ningún lenguaje supera el +13 %.

### 3.10 Interpretación de la Lección 3

**Relación con la pregunta analítica.**

1. **Sí hay tecnologías asociadas a un salario mayor:** Scala, Erlang, Elixir, Ruby, Go, F# y Rust entre los lenguajes, y Phoenix, Ruby on Rails y FastAPI entre los frameworks. PHP y sus frameworks (Laravel, Symfony), VBA y Dart están asociados a salarios menores.
2. **La región explica una parte de la diferencia.** Al comparar dentro de la región, las primas bajan, a veces a la mitad, pero en general conservan el signo.
3. **La diferencia cambia con la experiencia y la región.** Go es la prima más estable. Otras tecnologías pagan más al inicio (Bash/Shell, C++) o al final de la carrera (TypeScript, Java), y todas las primas son más amplias en Asia y África.
4. **La experiencia y la región pesan más que la tecnología.** El salario se triplica entre 0–2 y más de 20 años, y la mediana de América del Norte es 6,6 veces la de África. La mayor prima de un lenguaje es de alrededor de +29 %.

Son **asociaciones, no causas**. Un lenguaje bien pagado puede reflejar el tipo de empresa o de cargo. Estos resultados describen los datos y preparan las siguientes etapas del proyecto: visualización y modelado.